# 분류된 문장으로 교정 새롭게 실행

### 분류 결과 분포 확인하기

In [ ]:
import time
import pandas as pd
import numpy as np
import os
from google import genai
from google.genai import types
from tqdm import tqdm

In [ ]:
df1 = pd.read_csv("./output/router_full_test_results.csv")

In [ ]:
pd.crosstab(df1['is_correct'], df1['routed_model'])

확인 결과 해당 서비스 기간 고성능 모델로는 gpt 4.1, 경량 모델로는 gemini 2.0 flash를 사용했음을 확인함

## 실행 결과(smart routing -> llm 모델 입력)와 원본 정답 문장(sentencify.ai)과 비교하여 품질 측정하기

In [ ]:
# =========================================================
# [Mock API Functions] 
# =========================================================
def call_gemini_flash(text, intensity, field):
    # 실제로는 google.generativeai 라이브러리 사용
    # prompt = f"Correct this text: '{text}' with {intensity} intensity for {field} field."
    time.sleep(0.5) # API latency simulation
    return f"[Flash Correction] ({intensity}/{field}) {text} -> (Corrected by Flash)"

def call_gpt4(text, intensity, field):
    # 실제로는 openai 라이브러리 사용
    time.sleep(0.8) 
    return f"[GPT-4 Correction] ({intensity}/{field}) {text} -> (Corrected by GPT-4)"

In [ ]:
# =========================================================
# [Execution Loop] 라우팅 결과에 따라 모델 호출
# =========================================================
generated_results = []

print("🚀 [End-to-End Evaluation] 실제 모델 교정 시작...")

for index, row in tqdm(df1.iterrows(), total=len(df1)):
    # 입력 데이터 추출 (파싱된 태그 활용)
    input_text = row['input_sentence']
    intensity = row['tag_intensity']
    field = row['tag_field']
    target_model = row['routed_model'] # 'gemini-flash' or 'gpt-4'
    
    # 라우팅된 모델에 따라 API 호출
    try:
        if target_model == 'gemini-flash':
            output_text = call_gemini_flash(input_text, intensity, field)
        else:
            output_text = call_gpt4(input_text, intensity, field)
    except Exception as e:
        output_text = f"Error: {str(e)}"

    # 결과 저장
    generated_results.append(output_text)

# 데이터프레임에 결과 추가
df1['final_output'] = generated_results

In [ ]:
# =========================================================
# [저장] 최종 결과 (채점 전)
# =========================================================
df1.to_csv("./output/2.final_generation_results.csv", index=False, encoding='utf-8-sig')
print("💾 생성 결과 저장 완료: ./output/2.final_generation_results.csv")

# 결과 미리보기
display(df1[['input_sentence', 'routed_model', 'final_output']].head())